# IMPLEMENTASI PREPROCESSING DATA

Notebook ini digunakan untuk melakukan tahapan preprocessing pada dataset komentar Instagram yang telah diperoleh dari proses scraping. Tahapan preprocessing meliputi case folding, cleaning, tokenizing, normalisasi, dan stopword removal sebelum data digunakan pada proses klasifikasi sentimen menggunakan metode Multinomial Naïve Bayes.

In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\python.exe
3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]


In [2]:
import pandas as pd
import re

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [3]:
# Load dataset
df = pd.read_excel("../dataset/komentar_timnas.xlsx")
df.head()

,postingan,username,komentar
0,1,krnwnrzaa_,Era keganasan John Herdman ☠️
1,1,reza_erfit,JOHN HERDMAN OUT!!
2,1,yola_rizma74,Dihancurkan si patrick dibangun lagi john herd...
3,1,riandy_rians,Kalau pendapat saya ya kita berikan kesempatan...
4,1,bri.howardd,@dzakyardiyansah_ tetap aja john herdman baru ...


In [4]:
print("Jumlah data :", len(df))
print()
df.info()

Jumlah data : 3710

<class 'pandas.DataFrame'>
RangeIndex: 3710 entries, 0 to 3709
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   postingan  3710 non-null   int64
 1   username   3710 non-null   str  
 2   komentar   3710 non-null   str  
dtypes: int64(1), str(2)
memory usage: 336.9 KB


In [5]:
komentar = df["komentar"]
print("Jumlah awal :", len(komentar))
komentar = komentar.dropna()
print("Setelah hapus kosong :", len(komentar))
komentar = komentar.astype(str)

komentar = komentar[
    komentar.str.split().str.len() >= 2
]

print("Setelah hapus komentar pendek :", len(komentar))
komentar = komentar.drop_duplicates()
print("Setelah hapus duplikat :", len(komentar))

Jumlah awal : 3710
Setelah hapus kosong : 3710
Setelah hapus komentar pendek : 3534
Setelah hapus duplikat : 3507


In [6]:
# ==========================================
# CLEANING
# ==========================================

def cleaning(text):

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#\w+", "", text)

    text = re.sub(r"\d+", "", text)

    text = re.sub(
        r"(.)\1{2,}",
        r"\1",
        text
    )

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [7]:
contoh_cleaning = pd.DataFrame()

contoh_cleaning["Komentar Asli"] = komentar

contoh_cleaning["Cleaning"] = (
    contoh_cleaning["Komentar Asli"]
    .apply(cleaning)
)

contoh_cleaning

,Komentar Asli,Cleaning
0,Era keganasan John Herdman ☠️,Era keganasan John Herdman
1,JOHN HERDMAN OUT!!,JOHN HERDMAN OUT
2,Dihancurkan si patrick dibangun lagi john herd...,Dihancurkan si patrick dibangun lagi john herdman
3,Kalau pendapat saya ya kita berikan kesempatan...,Kalau pendapat saya ya kita berikan kesempatan...
4,@dzakyardiyansah_ tetap aja john herdman baru ...,tetap aja john herdman baru awal ngelatih masi...
...,...,...
3704,sama² coach🔥,sama coach
3706,Kasih yang terbaik setiap Pertandingan,Kasih yang terbaik setiap Pertandingan
3707,Timnas jelek,Timnas jelek
3708,john out,john out


In [8]:
## Case Folding: Tahap case folding dilakukan untuk mengubah seluruh karakter menjadi huruf kecil (lowercase) sehingga penulisan kata menjadi seragam.

def case_folding(text):
    return str(text).lower()

contoh_casefold = pd.DataFrame()
contoh_casefold["Cleaning"] = contoh_cleaning["Cleaning"]
contoh_casefold["Case Folding"] = (
    contoh_casefold["Cleaning"]
    .apply(case_folding)
)
contoh_casefold

,Cleaning,Case Folding
0,Era keganasan John Herdman,era keganasan john herdman
1,JOHN HERDMAN OUT,john herdman out
2,Dihancurkan si patrick dibangun lagi john herdman,dihancurkan si patrick dibangun lagi john herdman
3,Kalau pendapat saya ya kita berikan kesempatan...,kalau pendapat saya ya kita berikan kesempatan...
4,tetap aja john herdman baru awal ngelatih masi...,tetap aja john herdman baru awal ngelatih masi...
...,...,...
3704,sama coach,sama coach
3706,Kasih yang terbaik setiap Pertandingan,kasih yang terbaik setiap pertandingan
3707,Timnas jelek,timnas jelek
3708,john out,john out


In [9]:
# ==========================================
# TOKENIZING
# ==========================================

def tokenizing(text):
    return text.split()

contoh_token = pd.DataFrame()

contoh_token["Case Folding"] = (
    contoh_casefold["Case Folding"]
)

contoh_token["Tokenizing"] = (
    contoh_token["Case Folding"]
    .apply(tokenizing)
)

contoh_token

,Case Folding,Tokenizing
0,era keganasan john herdman,"[era, keganasan, john, herdman]"
1,john herdman out,"[john, herdman, out]"
2,dihancurkan si patrick dibangun lagi john herdman,"[dihancurkan, si, patrick, dibangun, lagi, joh..."
3,kalau pendapat saya ya kita berikan kesempatan...,"[kalau, pendapat, saya, ya, kita, berikan, kes..."
4,tetap aja john herdman baru awal ngelatih masi...,"[tetap, aja, john, herdman, baru, awal, ngelat..."
...,...,...
3704,sama coach,"[sama, coach]"
3706,kasih yang terbaik setiap pertandingan,"[kasih, yang, terbaik, setiap, pertandingan]"
3707,timnas jelek,"[timnas, jelek]"
3708,john out,"[john, out]"


In [10]:
# ==========================================
# NORMALISASI
# ==========================================

normalisasi = {

    "yg": "yang",
    "gk": "ga",
    "gak": "ga",
    "nggak": "ga",
    "tdk": "tidak",
    "gabisa": "tidak bisa",
    "jd": "jadi",
    "bgt": "banget",
    "utk": "untuk",
    "dr": "dari",
    "krn": "karena",
    "td": "td",
    "knp": "kenapa",
    "dgn": "dengan",
    "dg": "dengan",
    "aja": "saja",
    "udh": "sudah",
    "udah": "sudah",
    "tp": "tapi",
    "klo": "kalau",
    "kalo": "kalau",
    "smg": "semoga",
    "amiin": "amin",
    "aamiin": "amin",
    "aminnn": "amin",
    "gas": "ayo",
    "gass": "ayo",
    "gasss": "ayo",
    "gaskeun": "ayo",
    "gasken": "ayo",
    "gaskenn": "ayo",
    "jgn": "jangan",
    "dlu": "dulu",
    "mantaaap": "mantap",
    "mantapp": "mantap",
    "mantappp": "mantap",
    "kereeen": "keren",
    "kereen": "keren",
    "kerennn": "keren",
    "goks": "keren",
    "josss": "bagus",
    "gacor": "bagus",
    "mantul": "mantap",
    "herman": "herdman",
    "jon": "john",
    "gw": "saya",
    "gua": "saya",
    "gue": "saya",
    "lu": "kamu",
    "loe": "kamu",
    "lo": "kamu",
    "maen": "main",
    "mainnya": "main",
    "liat": "lihat",
    "nontonin": "nonton",
    "indo": "indonesia",
    "timnasindo": "timnas",
    "id": "indonesia",
    "yo": "ayo",
    "lets": "ayo",
    "go": "ayo",
    "gooo": "ayo",
    "goooo": "ayo",
    "gooooo": "ayo",
    "goodluck": "semoga sukses",
    "bg": "bang",
    "nyari": "cari",
    "ngelatih": "latih",
    "pelatihnya": "pelatih",
    "wkwkwk": "tertawa",
    "wkwkwkwk": "tertawa",
    "wkwwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "awokwok": "tertawa",
    "etam": "beckham",
    "etamm": "beckham",
    "tamm": "beckham",
    "bekam": "beckham",
    "beckam": "beckham",
    "rispek": "respect",
    "tp": "tapi",
    "pildun": "piala dunia",
    "napa": "kenapa",
    "emg": "memang",
    "emng": "memang",
    "gmn": "bagaimana",
    "gimana": "bagaimana",
    "kuch": "coach"
}

def normalisasi_text(tokens):
    
    return [
        normalisasi.get(kata, kata)
        for kata in tokens
    ]

# ==========================================
# CONTOH NORMALISASI
# ==========================================

contoh_normalisasi = pd.DataFrame()

contoh_normalisasi["Tokenizing"] = (
    contoh_token["Tokenizing"]
)

contoh_normalisasi["Normalisasi"] = (
    contoh_normalisasi["Tokenizing"]
        .apply(normalisasi_text)
)

contoh_normalisasi

,Tokenizing,Normalisasi
0,"[era, keganasan, john, herdman]","[era, keganasan, john, herdman]"
1,"[john, herdman, out]","[john, herdman, out]"
2,"[dihancurkan, si, patrick, dibangun, lagi, joh...","[dihancurkan, si, patrick, dibangun, lagi, joh..."
3,"[kalau, pendapat, saya, ya, kita, berikan, kes...","[kalau, pendapat, saya, ya, kita, berikan, kes..."
4,"[tetap, aja, john, herdman, baru, awal, ngelat...","[tetap, saja, john, herdman, baru, awal, latih..."
...,...,...
3704,"[sama, coach]","[sama, coach]"
3706,"[kasih, yang, terbaik, setiap, pertandingan]","[kasih, yang, terbaik, setiap, pertandingan]"
3707,"[timnas, jelek]","[timnas, jelek]"
3708,"[john, out]","[john, out]"


In [11]:
# ==========================================
# STOPWORD
# ==========================================

stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

# jangan hapus kata negasi
kata_penting = {
    "tidak",
    "bukan",
    "jangan",
    "belum"
}

stopwords = stopwords - kata_penting


def stopword_removal(tokens):

    return [
        kata
        for kata in tokens
        if kata not in stopwords
    ]

In [12]:
contoh_stopword = pd.DataFrame()

contoh_stopword["Normalisasi"] = (
    contoh_normalisasi["Normalisasi"]
)

contoh_stopword["Stopword Removal"] = (
    contoh_stopword["Normalisasi"]
        .apply(stopword_removal)
)

contoh_stopword

,Normalisasi,Stopword Removal
0,"[era, keganasan, john, herdman]","[era, keganasan, john, herdman]"
1,"[john, herdman, out]","[john, herdman, out]"
2,"[dihancurkan, si, patrick, dibangun, lagi, joh...","[dihancurkan, si, patrick, dibangun, john, her..."
3,"[kalau, pendapat, saya, ya, kita, berikan, kes...","[kalau, pendapat, berikan, kesempatan, coach, ..."
4,"[tetap, saja, john, herdman, baru, awal, latih...","[tetap, john, herdman, baru, awal, latih, cari..."
...,...,...
3704,"[sama, coach]","[sama, coach]"
3706,"[kasih, yang, terbaik, setiap, pertandingan]","[kasih, terbaik, pertandingan]"
3707,"[timnas, jelek]","[timnas, jelek]"
3708,"[john, out]","[john, out]"


## Preprocessing Seluruh Dataset

Setelah setiap tahapan preprocessing diuji menggunakan beberapa contoh data, seluruh komentar diproses menggunakan tahapan yang sama, yaitu cleaning, case folding, tokenizing, normalisasi, dan stopword removal. Hasil preprocessing kemudian disimpan ke dalam file Excel untuk digunakan pada proses pelatihan model.

In [13]:
# ==========================================
# PREPROCESSING SELURUH DATA
# ==========================================

hasil_preprocessing = []

for komen in komentar:

    # Case Folding
    casefold = case_folding(komen)

    # Cleaning
    cleaned = cleaning(casefold)

    # Tokenizing
    token = tokenizing(cleaned)

    # Normalisasi
    normal = normalisasi_text(token)

    # Stopword Removal
    stopword = stopword_removal(normal)

    hasil_preprocessing.append(
        " ".join(stopword)
    )

In [14]:
hasil_df = pd.DataFrame({
    "komentar": hasil_preprocessing
})

In [15]:
# Hapus komentar kosong
hasil_df = hasil_df[
    hasil_df["komentar"].str.strip() != ""
]

# Hapus komentar kurang dari 2 kata
hasil_df = hasil_df[
    hasil_df["komentar"].str.split().str.len() >= 2
]

# Hapus duplikat
hasil_df = hasil_df.drop_duplicates()

hasil_df.reset_index(drop=True, inplace=True)

In [16]:
hasil_df.to_excel(
    "../dataset/hasil_preprocessing.xlsx",
    index=False
)

print("Jumlah Data Setelah Preprocessing :", len(hasil_df))
print("File berhasil disimpan : hasil_preprocessing.xlsx")

Jumlah Data Setelah Preprocessing : 3008
File berhasil disimpan : hasil_preprocessing.xlsx


In [17]:
hasil_df

,komentar
0,era keganasan john herdman
1,john herdman out
2,dihancurkan si patrick dibangun john herdman
3,kalau pendapat berikan kesempatan coach john h...
4,tetap john herdman baru awal latih cari gaya p...
...,...
3003,sama coach
3004,kasih terbaik pertandingan
3005,timnas jelek
3006,john out


In [18]:
df = pd.read_excel("../dataset/hasil_preprocessing.xlsx")

# Menambahkan kolom sentimen setelah kolom komentar
df.insert(
    1,
    "sentimen",
    ""
)

# Simpan ke Excel
df.to_excel(
    "../dataset/label_manual.xlsx",
    index=False
)

print("File label_manual.xlsx berhasil dibuat.")

File label_manual.xlsx berhasil dibuat.
